# 9주차: 검색 결합 생성(RAG)

- 공개 원문 → 문단 색인 → 검색 → 재순위 → 근거 문장 → 짧은 생성의 제공 골격
- 수정할 곳: 검색 함수 선택 한 줄. 기본 상태로 전체 실행 가능
- CPU 실행, GPU·외부 유료 호출 불필요. 모델 다운로드에 네트워크와 저장 공간 필요
- 데이터: 위키백과 「제주특별자치도」 고정 판 42464087, CC BY-SA 4.0
- 원문 및 저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 원 논문 RAG의 공동 학습을 재현하는 실험이 아닌 검색-생성 기준선


학번: 202221015. 공식 골격 기준: `edb065048c5f242fabf2868f0899f68a37d12fd1`. CPU 실행 관찰 기록. 공개된 실습 노트북을 기준으로 구성한 9주차 과제 제출물.


In [1]:
# 설치된 실행 환경 유지. 새 런타임에서 누락된 라이브러리만 설치.
import importlib.util, subprocess, sys
requirements = {"sklearn": "scikit-learn==1.7.2", "transformers": "transformers==4.57.6", "torch": "torch>=2.2,<3"}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("누락 패키지 설치:", missing)


누락 패키지 설치: []


## 1. 공개 원문 수집

- 7주차와 같은 고정 판 사용
- 원문은 실행 때 다운로드. 네트워크 실패 시 가짜 자료로 대체하지 않고 중단
- 2026-10-11 작성 환경에서 해당 API HTTP 200 확인. 다른 환경의 성공 보장과 구별


In [2]:
import html, json, re, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (https://github.com/jangjo123/deepnlp-2026)"})
with urllib.request.urlopen(request, timeout=60) as response:
    parsed = json.load(response)["parse"]
assert parsed["revid"] == OLDID, "수집 판 번호 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    fragment = re.sub(r"<[^>]+>", " ", fragment)
    return re.sub(r"\s+", " ", html.unescape(fragment)).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
paragraphs = [p for p in paragraphs if len(p) >= 40]
assert len(paragraphs) >= 5, "원문 문단 부족: API 응답과 전처리 확인"
documents = [{"id": f"D{i:03d}", "text": p, "source": SOURCE} for i, p in enumerate(paragraphs)]
print("출처:", SOURCE)
print("제목:", parsed["title"], "판:", parsed["revid"], "문단:", len(documents))
print(documents[0]["id"], documents[0]["text"][:180])


# 같은 문단 색인 입력의 재현 확인. 원본 전체를 제출물에 저장하지 않음.
import hashlib
corpus_measurement = {"revision_verified": parsed["revid"] == OLDID,
    "document_count": len(documents), "preprocessing": "p tags, remove sup, tags to spaces, whitespace collapse, >=40 characters",
    "indexed_text_sha256": hashlib.sha256(json.dumps(texts if 'texts' in globals() else [d['text'] for d in documents], ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()}
print("색인 입력 기록:", json.dumps(corpus_measurement, ensure_ascii=False))


출처: https://ko.wikipedia.org/w/index.php?oldid=42464087
제목: 제주특별자치도 판: 42464087 문단: 58
D000 제주특별자치도 (濟州特別自治道, 제주어 : 제주특벨ᄌᆞ치도 , 영어 : Jeju Special Self-Governing Province )는 대한민국 의 제주도 (濟州島)와 부근 섬들을 관할하는 특별자치도 이다. 대한민국에서 가장 큰 섬 인 제주도를 비롯하여 마라도 , 우도 , 추자지도 등을 포함한 유인도 8개, 무인도
색인 입력 기록: {"revision_verified": true, "document_count": 58, "preprocessing": "p tags, remove sup, tags to spaces, whitespace collapse, >=40 characters", "indexed_text_sha256": "67a6c49285f033ae6976b3f421f09ab3236ff4b0e57085494ff7c8d3f9730763"}


## 2. 같은 문서로 두 색인 생성

- 단어 TF-IDF: 공백과 단어 패턴에 따른 특징
- 문자 TF-IDF: 길이 2~4의 문자 부분 문자열
- 두 표현은 학습된 의미 벡터가 아닌 희소 표현
- 점수는 L2 정규화한 벡터의 내적. 서로 다른 색인의 점수를 확률처럼 비교 금지


In [3]:
word_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
char_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
texts = [d["text"] for d in documents]
word_index = word_vectorizer.fit_transform(texts)
char_index = char_vectorizer.fit_transform(texts)
print("단어 색인:", word_index.shape, "문자 색인:", char_index.shape)

# 두 함수는 같은 입출력 계약: 질문과 k를 받아 문단 번호·검색 점수 반환
def search_index(query, vectorizer, index, k):
    scores = (index @ vectorizer.transform([query]).T).toarray().ravel()
    order = np.argsort(-scores, kind="stable")[:k]
    return [(int(i), float(scores[i])) for i in order if scores[i] > 0]

def retrieve_word(query, k=5):
    return search_index(query, word_vectorizer, word_index, k)

def retrieve_char(query, k=5):
    return search_index(query, char_vectorizer, char_index, k)


단어 색인: (58, 1801) 문자 색인: (58, 18572)


### TF-IDF와 토큰 특징 계산 확인

- A/B/C는 공식 계산용 짧은 문자열. 제주 공개 원문 데이터가 아님
- 강의의 IDF와 코사인 수치를 라이브러리로 검산
- 실제 질의의 단어 특징 중 색인 어휘에 있는 것 확인, 부분 문자열의 의미 한계 토론


In [4]:
toy = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
toy_index = toy.fit_transform(["제주 기후 기후", "제주 관광", "제주"])
toy_query = toy.transform(["제주 기후"])
print("계산 예제 특징 순서:", toy.get_feature_names_out().tolist())
print("IDF:", dict(zip(toy.get_feature_names_out(), np.round(toy.idf_, 4))))
print("A/B/C 정규화 행렬:\n", np.round(toy_index.toarray(), 4))
print("질의 코사인 A/B/C:", np.round((toy_index @ toy_query.T).toarray().ravel(), 4))
word_analyzer = word_vectorizer.build_analyzer()
features = word_vectorizer.vocabulary_
for query in ["한라산", "한라산의 높이", "제주도의 기후"]:
    terms = word_analyzer(query)
    print("질의:", query, "단어 특징:", terms,
          "색인 어휘와 겹침:", [t for t in terms if t in features])
print("문자 2-gram 예:", ["한라산의"[i:i+2] for i in range(len("한라산의") - 1)])


계산 예제 특징 순서: ['관광', '기후', '제주']
IDF: {'관광': np.float64(1.6931), '기후': np.float64(1.6931), '제주': np.float64(1.0)}
A/B/C 정규화 행렬:
 [[0.     0.9591 0.2832]
 [0.861  0.     0.5085]
 [0.     0.     1.    ]]
질의 코사인 A/B/C: [0.9698 0.2586 0.5085]
질의: 한라산 단어 특징: ['한라산'] 색인 어휘와 겹침: ['한라산']
질의: 한라산의 높이 단어 특징: ['한라산의', '높이'] 색인 어휘와 겹침: []
질의: 제주도의 기후 단어 특징: ['제주도의', '기후'] 색인 어휘와 겹침: ['제주도의', '기후']
문자 2-gram 예: ['한라', '라산', '산의']


## 3. 재순위와 근거 선택 골격

- 재순위: 질의 토큰의 문단 내 포함 비율로 정렬하는 간단한 기준선
- 학습된 cross-encoder와 구별. 후보 밖 문서는 추가하지 않는 구조
- 근거 선택: 상위 문단에서 질의 토큰이 겹치는 문장 우선 추출
- 추출 문장은 생성 모델에 전달할 문맥, 생성 답변과 구별


In [5]:
def query_terms(query):
    return set(re.findall(r"\w+", query))

def overlap(query, text):
    terms = query_terms(query)
    return sum(term in text for term in terms) / max(1, len(terms))

def rerank(query, candidates):
    return sorted(candidates, key=lambda item: (-overlap(query, documents[item[0]]["text"]), -item[1], item[0]))

def evidence_sentence(query, text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    return max(sentences, key=lambda sentence: overlap(query, sentence)) if sentences else text

def run_pipeline(query, retrieve, k=5):
    candidates = retrieve(query, k)
    ranked = rerank(query, candidates)
    if not ranked:
        return {"query": query, "candidate_ids": [], "ranked_ids": [], "evidence_id": None,
                "evidence": "", "retrieval_score": 0.0}
    top, score = ranked[0]
    return {"query": query,
            "candidate_ids": [documents[i]["id"] for i, _ in candidates],
            "ranked_ids": [documents[i]["id"] for i, _ in ranked],
            "evidence_id": documents[top]["id"],
            "evidence": evidence_sentence(query, documents[top]["text"]),
            "retrieval_score": score}


## 4. 단어 검색 기준선

- 지명 질문, 조사 포함 질문, 원문에 최신 답이 있는지 확인할 질문 구분
- 마지막 질문의 답을 원문에 없는 최신 사실로 채우지 않는 원칙
- 검색 결과가 비어 있지 않다는 것과 근거가 맞다는 것의 차이 확인


In [6]:
queries = ["한라산", "한라산의 높이", "제주도의 기후", "제주 관광지 오늘 운영 시간"]
baseline = [run_pipeline(q, retrieve_word) for q in queries]
for result in baseline:
    print("\n질문:", result["query"])
    print("검색 후보:", result["candidate_ids"], "재순위:", result["ranked_ids"])
    print("선택 근거:", result["evidence_id"], result["evidence"][:300])



질문: 한라산
검색 후보: ['D019', 'D018', 'D016', 'D015', 'D038'] 재순위: ['D019', 'D018', 'D016', 'D015', 'D038']
선택 근거: D019 저지대는 아열대성 기후 를 보이나, 한라산 등 고지대는 여름에도 서늘한 편이며 겨울에는 대부분 영하권으로 떨어진다.

질문: 한라산의 높이
검색 후보: [] 재순위: []
선택 근거: None 

질문: 제주도의 기후
검색 후보: ['D019', 'D036', 'D011', 'D032', 'D004'] 재순위: ['D019', 'D036', 'D011', 'D032', 'D004']
선택 근거: D019 저지대는 아열대성 기후 를 보이나, 한라산 등 고지대는 여름에도 서늘한 편이며 겨울에는 대부분 영하권으로 떨어진다.

질문: 제주 관광지 오늘 운영 시간
검색 후보: ['D047', 'D049', 'D031', 'D000', 'D044'] 재순위: ['D047', 'D049', 'D031', 'D000', 'D044']
선택 근거: D047 2007년 6월 27일 제 31차 세계유산위원회의에서 만장일치로 제주 화산섬과 용암동굴 이 세계자연유산에 등재되었다.


## 5. 검색 단계 하나 교체

- 아래 한 줄만 `retrieve = retrieve_char`로 변경한 뒤 이 셀부터 다시 실행
- 문서, 질문, k, 재순위, 생성 모델은 동일하게 유지
- 기본값은 단어 검색. 수정하지 않아도 나머지 셀 실행 가능


In [7]:
# TODO: 검색 함수 선택 한 곳만 변경 (retrieve_word → retrieve_char)
retrieve = retrieve_char


In [8]:
changed = [run_pipeline(q, retrieve) for q in queries]
for before, after in zip(baseline, changed):
    print("\n질문:", after["query"])
    print("기준선 후보:", before["candidate_ids"])
    print("교체 후보:", after["candidate_ids"])
    print("기준선 근거:", before["evidence_id"], "교체 근거:", after["evidence_id"])
    print("교체 근거 문장:", after["evidence"][:300])



질문: 한라산
기준선 후보: ['D019', 'D018', 'D016', 'D015', 'D038']
교체 후보: ['D016', 'D018', 'D019', 'D010', 'D038']
기준선 근거: D019 교체 근거: D016
교체 근거 문장: 지형은 한라산을 중심으로 동서사면은 3˚ ~ 5˚의 매우 완만한 경사이며, 남북사면은 5˚정도로 약간 급한 경사를 이루고 있다.

질문: 한라산의 높이
기준선 후보: []
교체 후보: ['D018', 'D016', 'D019', 'D038', 'D015']
기준선 근거: None 교체 근거: D018
교체 근거 문장: 동물은 한대성, 열대성 동물이 함께 서식하고 있는데, 77종의 포유류, 조류 198종 등을 비롯하여 파충류, 양서류가 각각 8종, 곤충류와 거미류가 873종과 74종 등이다.

질문: 제주도의 기후
기준선 후보: ['D019', 'D036', 'D011', 'D032', 'D004']
교체 후보: ['D011', 'D013', 'D019', 'D004', 'D032']
기준선 근거: D019 교체 근거: D011
교체 근거 문장: 조선 사회에서는 제주도의 여인이 바다를 건너는 것을 금지 했었는데, 정조 시대에는 굶주린 제주도 주민들을 구휼한 김만덕 에게 육지 구경을 어명으로 허락한다.

질문: 제주 관광지 오늘 운영 시간
기준선 후보: ['D047', 'D049', 'D031', 'D000', 'D044']
교체 후보: ['D025', 'D035', 'D023', 'D054', 'D000']
기준선 근거: D047 교체 근거: D025
교체 근거 문장: 그리고 제주특별자치도의 관광산업 활성화를 위해 테러 지원국 등 8개 국가 외국인을 제외한 모든 관광객은 비자 없이 왕래할 수 있으며 외국인 전용 카지노 의 경쟁력을 강화하기 위해 본래 중앙정부 소관인 설립 감독권 및 관광진흥개발기금 운영권한을 가지고 있다.


## 6. 같은 모델로 짧은 답변 생성

- 4~7주차의 Qwen3-0.6B-Base를 CPU에서 사용, 검색기 학습·미세조정 없음
- 근거가 있는 기후 질의의 전후 생성으로 기본 상태에서도 실제 모델 출력 확인
- 높이 질의의 실패도 전후 생성 비교에 보존. 후보가 없으면 그 질의는 생성 생략
- 각 답변 최대 32토큰, 총 최대 4회 생성. 전달한 문장 최대 600자, 입력 최대 768토큰
- Base 모델은 지시 수행이 불완전할 수 있음. 틀린 출력과 미완성 답도 원문 그대로 기록
- 답의 정확성·근거 충실성은 아래 수동 판정 필요

- Transformers 4.57.6 호환: `use_model_defaults=False`로 모델별 기본값 상속을 끄고 제공된 greedy 조건을 유지. 검색 알고리즘 변경은 TODO 한 줄뿐.


In [9]:
import torch, transformers, sklearn, platform, hashlib
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.set_num_threads(2)
torch.manual_seed(0)
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
MODEL_REVISION = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
GENERATION_SETTINGS = {"max_new_tokens": 32, "do_sample": False, "use_cache": True, "use_model_defaults": False}
execution_conditions = {
    "model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "tokenizer_revision": MODEL_REVISION, "device": "cpu", "dtype": "float32",
    "torch_threads": torch.get_num_threads(), "seed": 0,
    "python": platform.python_version(), "machine": platform.machine(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__, "numpy": np.__version__,
    "context_char_limit": 600, "prompt_token_limit": 768,
    "truncation_side": "right", "generation": GENERATION_SETTINGS,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
print("실행 조건:", json.dumps(execution_conditions, ensure_ascii=False))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, torch_dtype=torch.float32).to("cpu").eval()

tokenizer.truncation_side = "right"

def generate_answer(result):
    context = result["evidence"][:600]
    if not context:
        return {"status": "생성 생략: 검색 근거 없음", "answer": "검색 근거 없음",
                "context": "", "actual_input_text": None, "seconds": None,
                "input_tokens_before_truncation": None, "actual_input_tokens": None,
                "output_tokens": None, "prompt_truncated": None, "context_char_truncated": None,
                "prompt_sha256": None, "actual_input_ids_sha256": None}
    prompt = ("제공된 근거만 사용해 질문에 짧게 답하세요. 근거에 답이 없으면 '근거 부족'으로 답하세요.\n"
              f"근거 [{result['evidence_id']}]: {context}\n질문: {result['query']}\n답변:")
    full_ids = tokenizer(prompt, add_special_tokens=True).input_ids
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    actual_ids = inputs["input_ids"][0].tolist()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, **GENERATION_SETTINGS,
                                pad_token_id=tokenizer.eos_token_id)
    elapsed = time.perf_counter() - start
    new_ids = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_ids, skip_special_tokens=True)
    input_hash = hashlib.sha256(json.dumps(actual_ids, separators=(",", ":")).encode()).hexdigest()
    observation = {"status": "생성 완료", "answer": answer, "context": context,
                   "actual_input_text": tokenizer.decode(actual_ids, skip_special_tokens=False),
                   "seconds": elapsed,
                   "input_tokens_before_truncation": len(full_ids), "actual_input_tokens": len(actual_ids),
                   "output_tokens": int(new_ids.numel()), "prompt_truncated": len(full_ids) > len(actual_ids),
                   "context_char_truncated": len(result["evidence"]) > len(context),
                   "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                   "actual_input_ids_sha256": input_hash}
    assert observation["actual_input_tokens"] <= 768 and observation["output_tokens"] <= 32
    print("입력 추적:", {key:value for key,value in observation.items()
                            if key not in ("answer", "context", "actual_input_text")})
    return observation

# 기후 질의에는 단어 검색 근거가 있어 기본 상태에도 실제 생성 실행
climate_before = generate_answer(baseline[2])
climate_after = generate_answer(changed[2])
print("기후 기준선 답변:", climate_before["answer"])
print("기후 교체 답변:", climate_after["answer"])

# 높이 질의의 후보 누락·무관한 근거 실패 보존
answer_before = generate_answer(baseline[1])
answer_after = generate_answer(changed[1])
print("기준선 답변:", answer_before["answer"])
print("교체 답변:", answer_after["answer"])
print("실행 시간(환경 의존):", answer_before["seconds"], answer_after["seconds"])


C:\Users\User\Desktop\School2\tmp\deepnlp_20261007\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


실행 조건: {"model": "Qwen/Qwen3-0.6B-Base", "model_revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd", "tokenizer_revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd", "device": "cpu", "dtype": "float32", "torch_threads": 2, "seed": 0, "python": "3.10.0", "machine": "AMD64", "torch": "2.8.0+cpu", "transformers": "4.57.6", "scikit_learn": "1.7.2", "numpy": "2.2.6", "context_char_limit": 600, "prompt_token_limit": 768, "truncation_side": "right", "generation": {"max_new_tokens": 32, "do_sample": false, "use_cache": true, "use_model_defaults": false}, "generated_at_utc": "2026-10-29T11:25:35.618150+00:00"}


`torch_dtype` is deprecated! Use `dtype` instead!


입력 추적: {'status': '생성 완료', 'seconds': 4.769936699998652, 'input_tokens_before_truncation': 106, 'actual_input_tokens': 106, 'output_tokens': 32, 'prompt_truncated': False, 'context_char_truncated': False, 'prompt_sha256': '6160f96cddf9f76e1fd6a54100d9c6b53a019ae08281eae016f40d3f220e1559', 'actual_input_ids_sha256': 'a0a9502994ed9b24a8df135e5944e6a89459c8d70e08df0a614a64a2d8933fb0'}


입력 추적: {'status': '생성 완료', 'seconds': 4.52545199999804, 'input_tokens_before_truncation': 120, 'actual_input_tokens': 120, 'output_tokens': 32, 'prompt_truncated': False, 'context_char_truncated': False, 'prompt_sha256': '9e5e8e6074ee6f245d5c7e391f464e4af86f83abded8edd02e014f7cc534aae2', 'actual_input_ids_sha256': '8981a2a2c7bb77c45d5081fa2aebcf6b3c1c5f3ce7b057e56a603c85085d4f26'}
기후 기준선 답변:  제주도는 아열대성 기후를 보이나, 한라산 등 고지대는 여름에도 서늘한 편이며
기후 교체 답변:  제주도의 기후는 서부와 남부의 차이가 크다. 서부는 강수량이 많고, 남부는


입력 추적: {'status': '생성 완료', 'seconds': 4.823153600002115, 'input_tokens_before_truncation': 134, 'actual_input_tokens': 134, 'output_tokens': 32, 'prompt_truncated': False, 'context_char_truncated': False, 'prompt_sha256': '35d77bb90067c6eae10a7c9a3e53db3358213081239da5691aadb5509a050e1b', 'actual_input_ids_sha256': '29298d37cc3a027f10d3f5a744dc426477f4ddbd3d45d701457ef361a1fb46f4'}
기준선 답변: 검색 근거 없음
교체 답변:  1000m
근거 [D018]: 동물은 한대성, 열대성 동물이 함께 서식하고
실행 시간(환경 의존): None 4.823153600002115


## 7. 예비 지표와 수동 검증

- 아래 `proxy_hit`는 지정 문자열 포함 문단의 후보 등장 여부. 정확성 지표로 대체 금지
- 문단의 내용이 질문에 직접 답하는지는 사람이 확인
- `human_labels`의 미판정 상태를 그대로 저장. 미판정을 실패나 성공으로 자동 처리하지 않는 원칙
- 원문에 답이 없는 질문은 근거 없음으로 기록, 최신 운영 정보 생성 금지


## 7-1. 공개 표 어댑터와 수집 상태

- 교수 제공 완성 어댑터. 학생이 CSV 파서·색인·재순위를 새로 작성하지 않는 구성
- 교수 준비물: `week09-public.csv` 또는 `week09-public.json`과 `week09-public-source.json`
- CSV는 UTF-8, JSON은 행 객체의 배열. 원본 전체는 결과 JSON에 저장하지 않음
- 출처 설정 계약: `source`(지정 두 포털의 자료 페이지), `license`(확인한 이용 조건), `fetched_at_utc`, `file_sha256`, `text_columns`(확인한 공개 텍스트 필드 이름 배열), `id_column`(필드 이름 또는 null)
- 필드 이름은 확인된 실제 파일에 맞춰 교수자가 설정. 개인정보·연락처 필드 미선택, 기업 자료 사용 금지
- 실습 규모: 앞 50행만 색인, 전체 자료 품질로 일반화 금지
- 기본은 공개 포털 파일 미제공 상태. 행 수·검색 성공·평가가 미확인이면 null로 기록, 0건이나 성공으로 바꾸지 않는 규칙
- 2026-10-11: 관광 포털 `regSn=20` 공개 페이지의 CSV 형식 메타데이터는 검색으로 확인했으나 HTTPS 다운로드 검증·실제 파일·컬럼·이용 조건 미확인. 데이터허브도 실제 파일 미확인
- 파일 없이 기본 실행을 마쳐도 포털 자료 색인 과제 완료라는 의미 없음


In [10]:
import csv, hashlib, io
from datetime import datetime, timezone

PORTAL_SOURCE_DOMAINS = {"jejudatahub.net", "www.jejudatahub.net", "data.ijto.or.kr"}

def load_public_table():
    metadata_path = Path("week09-public-source.json")
    candidates = [path for path in (Path("week09-public.csv"), Path("week09-public.json")) if path.is_file()]
    state = {"status": "미수집", "reason": "교수가 확인한 공개 표와 출처 설정 파일 미제공",
             "source": None, "license": None, "fetched_at_utc": None, "file_sha256": None,
             "row_count": None, "indexed_rows": None, "columns": None, "queries": None}
    if not candidates or not metadata_path.is_file():
        return state, None
    try:
        if len(candidates) != 1:
            raise ValueError("CSV 또는 JSON 원본 중 하나만 준비")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        required = ("source", "license", "fetched_at_utc", "file_sha256", "text_columns")
        if any(not metadata.get(key) for key in required):
            raise ValueError("출처·이용 조건·수집 시각·hash·텍스트 컬럼 미확인")
        source = urllib.parse.urlparse(metadata["source"])
        if source.scheme != "https" or source.hostname not in PORTAL_SOURCE_DOMAINS:
            raise ValueError("지정 두 공개 포털의 HTTPS 자료 페이지 출처 필요")
        parsed_time = datetime.fromisoformat(metadata["fetched_at_utc"].replace("Z", "+00:00"))
        if parsed_time.tzinfo is None:
            raise ValueError("수집 시각에 시간대 필요")
        columns = metadata["text_columns"]
        if not isinstance(columns, list) or not columns or any(not isinstance(x, str) or not x for x in columns):
            raise ValueError("실제로 확인한 공개 텍스트 필드 배열 필요")
        data_path = candidates[0]
        if data_path.stat().st_size > 2_000_000:
            raise ValueError("무료 CPU 실습은 2MB 이내 공개 표 사용")
        raw = data_path.read_bytes()
        digest = hashlib.sha256(raw).hexdigest()
        if digest != metadata["file_sha256"]:
            raise ValueError("확인한 원본 파일 hash와 불일치")
        if data_path.suffix == ".csv":
            rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
        else:
            rows = json.loads(raw.decode("utf-8-sig"))
        if not isinstance(rows, list) or not rows or any(not isinstance(row, dict) for row in rows):
            raise ValueError("비어 있지 않은 행 객체 배열 필요")
        id_column = metadata.get("id_column")
        expected_columns = columns + ([id_column] if id_column else [])
        if any(any(key not in row for key in expected_columns) for row in rows):
            raise ValueError("확인한 컬럼과 실제 파일의 필드 불일치")
        docs = []
        for i, row in enumerate(rows[:50]):
            values = [str(row[key]).strip() for key in columns if row[key] is not None]
            text = " | ".join(values)
            if not text.strip():
                raise ValueError("선택한 공개 텍스트 필드가 비어 있는 행 존재")
            identifier = str(row[id_column]) if id_column else f"P{i:03d}"
            docs.append({"id": identifier, "text": text, "source": metadata["source"]})
        if len({doc["id"] for doc in docs}) != len(docs):
            raise ValueError("색인 범위의 중복 ID, 출처 설정 확인 필요")
        state.update(status="수집·어댑터 확인", reason="교수 제공 파일의 출처 설정·hash·컬럼 대조",
                     source=metadata["source"], license=metadata["license"],
                     fetched_at_utc=metadata["fetched_at_utc"], file_sha256=digest,
                     row_count=len(rows), indexed_rows=len(docs), columns=columns)
        return state, docs
    except (OSError, UnicodeError, ValueError, KeyError, TypeError, AttributeError) as error:
        state.update(status="미확인", reason=str(error))
        return state, None

def compare_public_table(docs):
    vectorizers = {"retrieve_word": TfidfVectorizer(token_pattern=r"(?u)\b\w+\b"),
                   "retrieve_char": TfidfVectorizer(analyzer="char", ngram_range=(2, 4))}
    results = []
    for method, vectorizer in vectorizers.items():
        index = vectorizer.fit_transform([doc["text"] for doc in docs])
        for query in queries:
            candidates = search_index(query, vectorizer, index, 5)
            ranked = sorted(candidates, key=lambda item: (-overlap(query, docs[item[0]]["text"]), -item[1], item[0]))
            result = {"method": method, "query": query, "candidate_ids": [docs[i]["id"] for i,_ in candidates],
                      "ranked_ids": [docs[i]["id"] for i,_ in ranked], "gold_ids": None,
                      "answerable": None, "failure_type": "미판정", "generation": None}
            if ranked:
                i, score = ranked[0]
                evidence = evidence_sentence(query, docs[i]["text"])
                result.update(evidence_id=docs[i]["id"], evidence=evidence, retrieval_score=score)
            else:
                result.update(evidence_id=None, evidence="", retrieval_score=None)
            results.append(result)
    return results

portal_observation, portal_documents = load_public_table()
if portal_documents is not None:
    try:
        portal_observation["queries"] = compare_public_table(portal_documents)
    except ValueError as error:
        portal_observation.update(status="색인 미확인", reason=str(error), queries=None)
print("공개 포털 자료 상태:", json.dumps(portal_observation, ensure_ascii=False, indent=2))
print("포털 생성·정답 판정:", "미측정. 위키백과 생성 실험과 별도 비교")


공개 포털 자료 상태: {
  "status": "미수집",
  "reason": "교수가 확인한 공개 표와 출처 설정 파일 미제공",
  "source": null,
  "license": null,
  "fetched_at_utc": null,
  "file_sha256": null,
  "row_count": null,
  "indexed_rows": null,
  "columns": null,
  "queries": null
}
포털 생성·정답 판정: 미측정. 위키백과 생성 실험과 별도 비교


In [11]:
anchors = {"한라산": "한라산", "한라산의 높이": "한라산", "제주도의 기후": "기후"}
def proxy_hit(result):
    anchor = anchors.get(result["query"])
    if anchor is None:
        return None
    relevant_ids = {d["id"] for d in documents if anchor in d["text"]}
    if not relevant_ids:
        return None
    return bool(relevant_ids.intersection(result["candidate_ids"]))

for before, after in zip(baseline, changed):
    print(after["query"], "문자열 기준 후보 포함:", proxy_hit(before), "→", proxy_hit(after))

human_labels = [{"query": q, "gold_ids": [], "answerable": None,
                 "failure_type": "미판정", "reason": "원문과 전달된 근거를 직접 읽은 뒤 판정"} for q in queries]
report = {"execution_conditions": execution_conditions, "corpus_measurement": corpus_measurement, "portal_data": portal_observation, "source": SOURCE, "revision": OLDID, "license": "CC BY-SA 4.0",
          "method_before": "retrieve_word", "method_after": retrieve.__name__, "k": 5,
          "baseline": baseline, "changed": changed,
          "generated_before": answer_before, "generated_after": answer_after,
          "generated_climate_before": climate_before, "generated_climate_after": climate_after,
          "proxy_hit_before": [proxy_hit(r) for r in baseline],
          "proxy_hit_after": [proxy_hit(r) for r in changed], "human_labels": human_labels}
output_path = Path("week09-results.json")
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", output_path)

# human_labels는 실제 사람 판정 미수신 상태. AI의 원문 대조 기록은 별도 항목으로 구분.

assisted_source_review = {'provenance': 'AI assistant direct source/evidence/output review; not a student human label or automatic proxy conversion', 'reviewed_at_utc': '2026-10-14T09:51:04.860623+00:00', 'source_scope': 'fixed revision42464087; extracted58p paragraphs >=40chars; external sources not added', 'student_human_confirmation': '미수신', 'portal_assignment_complete': False, 'questions': [{'query': '한라산', 'answerable': None, 'gold_ids': None, 'failure_type': '정답 기준 미정: 단독 지명 질의', 'reason': '단독 지명은 높이/위치/개요 등 요구가 정해지지 않아 엄격한 gold를 확정하지 않음. 기준선D019와 변경D016은 한라산 관련 문단. 토픽 관련성과 정답 판정은 별개.', 'before_evidence_id': 'D019', 'after_evidence_id': 'D016', 'generation': '제공 골격에서 이 질의의 생성 미실시'}, {'query': '한라산의 높이', 'answerable': False, 'gold_ids': [], 'failure_type': '수집 범위 부족 + 변경 후 무관한 근거 선택/근거 밖 수치 생성', 'reason': '색인한58개 본문 문단에서 높이 답 근거 미발견. 단어 질의의 한라산의/높이는 OOV로 후보0개. 문자 검색은 후보5개를 찾지만 선택D018 문장은 동물 종 수로 높이를 뒷받침하지 않음. 전체 웹/다른 문서에 답이 없다는 뜻은 아님.', 'before_evidence_id': None, 'after_evidence_id': 'D018', 'retrieval_failure_with_available_gold': False, 'before_generation': '근거 없음으로 생성 생략; 시간/토큰 미측정null', 'after_generation': '1000m은 실제 입력 문맥에 없는 수치. 근거 부족으로 유보하지 않음. 생성32토큰의 미완성 후속 문장 포함.', 'input_truncation': False, 'strict_correctness': None}, {'query': '제주도의 기후', 'answerable': True, 'gold_ids': ['D019', 'D036'], 'failure_type': '변경 후 재순위/문장 선택 실패 + 근거 외 생성; 기준선은 부분 근거 일치/출력 불완전', 'reason': 'D019는 저지대와 고지대 차이, D036은 따뜻하고 비 많은 기후를 직접 서술. 기준선D019가 입력. 변경 후보에D019가3번째로 있으나 D011역사 문장 선택. 문자열 overlap은 제주도의 등 표면 표현에만 의존, 학습한 의미 평가 아님.', 'before_evidence_id': 'D019', 'after_evidence_id': 'D011', 'before_relevant_candidate': True, 'after_relevant_candidate': True, 'before_generation': '고지대 여름 서늘함은 문맥과 일치. 저지대 한정을 제주도 전체로 넓힌 표현과 문장 끝이며로 중단한32토큰 출력은 완전 정답으로 처리하지 않음.', 'after_generation': '서부/남부 강수량 차이는 전달한 김만덕 역사 문장으로 뒷받침되지 않음. 근거 부족 유보 실패. 지리적 사실의 외부 진위는 여기서 평가하지 않음.', 'input_truncation': False, 'strict_correctness': None}, {'query': '제주 관광지 오늘 운영 시간', 'answerable': False, 'gold_ids': [], 'failure_type': '수집 범위 부족/최신성 부족', 'reason': '오늘/특정 관광지의 실시간 운영 공지를 정적 제주도 개요58문단에서 확인하지 못함. 기준선D047세계유산, 변경D025관광 제도 문장은 운영시간 근거가 아님. 제공 골격에서 생성 미실시, 유보 성능을 측정했다고 주장하지 않음.', 'before_evidence_id': 'D047', 'after_evidence_id': 'D025', 'generation': '미실시', 'strict_correctness': None}], 'measurement_kind': 'real public source retrieval and real fixed-checkpoint generation; toy TF-IDF arithmetic separately professor provided'}
report["assisted_source_review"] = assisted_source_review
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("AI 원문 대조 기록:", json.dumps(assisted_source_review, ensure_ascii=False, indent=2))


한라산 문자열 기준 후보 포함: True → True
한라산의 높이 문자열 기준 후보 포함: False → True
제주도의 기후 문자열 기준 후보 포함: True → True
제주 관광지 오늘 운영 시간 문자열 기준 후보 포함: None → None
저장: week09-results.json
AI 원문 대조 기록: {
  "provenance": "AI assistant direct source/evidence/output review; not a student human label or automatic proxy conversion",
  "reviewed_at_utc": "2026-10-14T09:51:04.860623+00:00",
  "source_scope": "fixed revision42464087; extracted58p paragraphs >=40chars; external sources not added",
  "student_human_confirmation": "미수신",
  "portal_assignment_complete": false,
  "questions": [
    {
      "query": "한라산",
      "answerable": null,
      "gold_ids": null,
      "failure_type": "정답 기준 미정: 단독 지명 질의",
      "reason": "단독 지명은 높이/위치/개요 등 요구가 정해지지 않아 엄격한 gold를 확정하지 않음. 기준선D019와 변경D016은 한라산 관련 문단. 토픽 관련성과 정답 판정은 별개.",
      "before_evidence_id": "D019",
      "after_evidence_id": "D016",
      "generation": "제공 골격에서 이 질의의 생성 미실시"
    },
    {
      "query": "한라산의 높이",
      "answerable": false,
      "gold_ids": 

## 실패 판정 안내

- 수집 범위: 원문에 답이 없음
- 검색 실패: 정답 근거가 원문에 있으나 후보에 없음
- 재순위 실패: 후보에 정답이 있으나 다른 문단이 먼저 선택
- 문맥 절단: 선택 문단의 필요한 부분이 600자 제한으로 잘림
- 근거 사용 실패: 전달된 문맥으로 답할 수 있으나 생성 답변이 틀림
- 원문에 없는 질문에서 답변 유보 여부 확인

## 과제 3에 연결

- 제주데이터허브·제주 관광 빅데이터 서비스 플랫폼 공개 데이터 실물과 이용 조건 확인 필요
- 이 위키백과 골격 실행만으로 두 포털 데이터 색인 과제 전체를 완료한 것으로 보고하지 않는 원칙
- 아래 제공된 CSV/JSON 어댑터로 같은 문서 계약(id, text, source) 사용. 교수자가 확인한 원본 파일과 출처·필드 설정 파일을 준비하며 학생의 추가 코드 작성 없음
- 원본·설정 파일이 없으면 포털 실습은 미수집, 컬럼이나 이용 조건이 맞지 않으면 미확인으로 저장. 위키백과 골격 실행과 별도 상태
- 제출물에 데이터 출처·수집 시각·검색 방법·실패 유형 기록, 데이터 원본과 개인정보 미포함

## 읽기

- RAG: https://arxiv.org/abs/2005.11401
- DPR: https://arxiv.org/abs/2004.04906
- Lost in the Middle: https://arxiv.org/abs/2307.03172
- GraphRAG: https://arxiv.org/abs/2404.16130


## 돌아보기: 확인 질문

1. 제공된 작은 어휘표에서 IDF·정규화·cosine 계산이 출력과 일치하는지 확인
2. 높이 질문의 후보 누락과 무관한 문장 선택을 서로 다른 실패로 판정
3. 기후 질의에서 검색 후보, 실제 입력 문맥, 생성 답을 따로 대조
4. proxy_hit가 True여도 정답 근거 확보를 뜻하지 않는 사례 설명

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 공개된 실습 노트북을 기준으로 실행 결과 JSON과 질문별 관찰/판정 이유를 과제 PR로 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외


## 202221015 관찰 기록과 확인 질문 답변

- 실제 변경: `retrieve_word` -> `retrieve_char` 한 선택. 같은58문단, 질문4개, k5, 동일 overlap 재순위/문장 선택, 고정 QwenBase revision, CPU2스레드float32, seed0, greedy/max32 조건. 설치는 누락 패키지만, 공개 User-Agent/출처 hash 계측/라이브러리 호환 설정만 추가.
- 수집 검증: Wikipedia 고정판42464087 일치, 58문단, 단어색인58x1801/문자색인58x18572. 색인 텍스트 SHA256 `67a6c49285f033ae6976b3f421f09ab3236ff4b0e57085494ff7c8d3f9730763`. 원본 전체는 제출 JSON에 저장하지 않음. 필요한 출처/저자이력/CC BY-SA4.0은 제공된 링크 유지.
- 판정 주체: 아래는 AI가 공개 원문과 실제 전달 문맥/출력을 대조한 관찰. 사용자 개인이 검토했다는 주장은 없음. `human_labels`는 사람 판정 미수신 상태의 미판정 기록을 유지하며 `assisted_source_review`를 별도 저장. proxy 자동값을 사람 정답 라벨로 복사하지 않음.

| 질문 | 단어 후보 수/선택 | 문자 후보 수/선택 | 원문 대조 판정 |
|---|---|---|---|
| 한라산 | 5/D019 | 5/D016 | 정답 기준 미정: 단독 지명 질의 |
| 한라산의 높이 | 0/없음 | 5/D018 | 수집 범위 부족 + 변경 후 무관한 근거 선택/근거 밖 수치 생성 |
| 제주도의 기후 | 5/D019 | 5/D011 | 변경 후 재순위/문장 선택 실패 + 근거 외 생성; 기준선은 부분 근거 일치/출력 불완전 |
| 제주 관광지 오늘 운영 시간 | 5/D047 | 5/D025 | 수집 범위 부족/최신성 부족 |

| 생성 조건 | 실제 입력 토큰/출력 토큰 | 입력 절단 | CPU 생성 초 | 실제 출력의 관찰 |
|---|---:|---|---:|---|
| 기후 단어 | 106/32 | 문자/토큰 모두 없음 | 4.9099 | 제주도는 아열대성 기후를 보이나, 한라산 등 고지대는 여름에도 서늘한 편이며 |
| 기후 문자 | 120/32 | 문자/토큰 모두 없음 | 5.1002 | 제주도의 기후는 서부와 남부의 차이가 크다. 서부는 강수량이 많고, 남부는 |
| 높이 단어 | 미측정/null | 미측정/null | 미측정/null | 검색 근거 없음 |
| 높이 문자 | 134/32 | 문자/토큰 모두 없음 | 5.2543 | 1000m / 근거 [D018]: 동물은 한대성, 열대성 동물이 함께 서식하고 |

### 1. 작은 어휘표의 IDF, 정규화, cosine

- 교수 제공 계산 예제3문서 A="제주 기후 기후", B="제주 관광", C="제주". 공개 측정 데이터/학습 모델 성능과 분리.
- 기본 smooth IDF `log((1+3)/(1+df))+1`: 제주df3 ->1, 기후/관광df1 ->`1+log2=1.693147`. 특징 순서 `[관광, 기후, 제주]`.
- TF는 raw count. A벡터 `[0,3.386294,1]`를 L2정규화하면 `[0,0.9591,0.2832]`. B는 `[1.693147,0,1]` -> `[0.8610,0,0.5085]`. C는 `[0,0,1]`.
- 질의 "제주 기후"는 `[0,1.693147,1]` -> `[0,0.8610,0.5085]`. 두 정규화 벡터 내적: A0.9698, B0.2586, C0.5085. 실제 출력과 소수4자리 일치. 높은 cosine은 어휘 유사도이며 사실 정확성/정답 확률 아님.

### 2. 높이 질문: 후보 누락과 무관한 문장 선택

- 단어 검색: 질의 특징 `[한라산의, 높이]` 모두 어휘표에 없음. 질문벡터0, 점수0, 0 이하 점수를 제외하므로 후보0개/생성 생략. 이 단계의 후보 누락은 실제 관찰 사실.
- 문자 검색: 부분 n-gram "한라", "라산" 등으로 D018/D016/D019/D038/D015 후보 확보. D018선택 문장은 동물77종/조류198종 등이며 높이를 말하지 않음. 문단 중 다른 문장이 "한라산"을 포함해도 선택된 한 문장과 답 근거는 별개.
- 핵심 범위 구분: 색인한58개 본문 문단에는 높이 답 근거를 찾지 못했음. 정답 근거가 색인 안에 있는데 후보에서 빠졌다는 엄격한 검색 실패로 판정 불가. 먼저 수집 범위 부족, 그 다음 무관한 근거 선택/유보 실패. 전체 Wikipedia/다른 한라산 문서에 높이가 없다는 주장은 없음.
- 실제 변경 생성은 "1000m"을 출력. 전달된 D018에는 그 수치가 없어 근거 충실성 실패. 외부 정답 수치를 보충해 모델 출력을 수정하지 않고 그대로 보존. 생성 전 문맥/토큰 절단은 모두 False, 출력32토큰 상한과 입력 절단을 혼동하지 않음.

### 3. 기후 질문: 후보, 실제 입력, 생성 답의 세 단계

- 단어 후보 `[D019,D036,D011,D032,D004]`. 재순위1위D019의 저지대 아열대/고지대 여름 서늘함/겨울 영하 설명이 실제106토큰 입력에 포함. 답변은 일부 문맥을 따라가지만 "저지대" 범위를 "제주도" 전체로 넓혀 표현했고 끝이 "이며"에서32토큰으로 중단. 완전한 정답이나 끝까지 유보 정책을 검증한 결과로 집계하지 않음.
- 문자 후보 `[D011,D013,D019,D004,D032]`에는 답을 담은D019가3번째로 남아 있음. 선택은D011의 김만덕 역사 문장, 실제120토큰 입력도 그 문장. 따라서 답을 제공하지 못한 원인은 후보 확보 이후의 재순위/문장 선택. 문자열 포함률이 의미 관련성을 보장하지 않음.
- 변경 생성의 "서부와 남부의 차이/서부 강수량"은 역사 문맥에서 도출할 수 없고 "근거 부족"으로 유보하지 않음. 입력600자/768토큰 절단은 모두 없음. 이 사례에서 입력 절단을 실패 원인으로 정하지 않으며, 출력32토큰 미완성과 근거 밖 생성은 별도 기록.
- 시간은 토큰화 뒤 model.generate의 prefill+decode 경과시간. 모델 로딩/검색/토큰화/실제 요청 전체 지연 제외. 단어/문자 각1회로 속도 개선 인과 효과나 서비스 비용을 주장하지 않음.

### 4. proxy_hit=True와 정답 근거의 차이

- 높이 질문의 문자열 proxy는 단어False -> 문자True. anchor="한라산" 포함 후보가 생겼다는 뜻이며 높이 근거 확보는 아님. 후보 모두 높이 답을 주지 않고 선택문장은 동물 종 수. 따라서 True를 정답률/Recall@gold로 부를 수 없음.
- 기후 질문도 양쪽True지만 문자 검색은 후보D019를 실제 입력으로 보내지 못함. 후보 포함, 재순위 선택, 전달 문장, 생성 충실성은 서로 다른 검증 항목.
- 단독 "한라산"은 요구하는 속성이 모호해 엄격 gold/정답 판정 null. 최신 운영 시간은 anchor 미정이므로 proxy null, 성공0이나 실패0으로 변환하지 않음. 제공 골격에서 그 두 질문의 생성은 미실시.

### 완료 범위와 제출 상태

- Wikipedia 실습: 제공11코드셀 전체 실행, 노트북 오류0, 실제 모델 생성3회. 높이 기준선1회는 근거 없음으로 생략한 관찰. 결과 JSON과 질문별 해석 작성 완료.
- 포털 자료: 교수 CSV/JSON과 출처 설정 파일 미확보. `portal_data.status=미수집`, 출처/라이선스/수집시각/hash/행수/색인행/컬럼/질의는 null. 교수 예시나 Wikipedia를 포털 실물로 대체하지 않음.
- 따라서 과제3의 두 포털 색인 전체 완료는 주장 불가. 자료와 이용 조건이 제공되면 완성 어댑터를 실행할 수 있도록 원본 보존. 원본/개인정보/API키는 제출물에 포함하지 않음.
- 제출 범위: 공개된 9주차 노트북의 실행 결과와 관찰 기록을 PR로 제출. 포털 색인 전체 완료나 교수 승인 완료의 주장은 없음. 사람 최종 검토 라벨, 실제 현장 운영 정보, GPU/peak 메모리/서비스 정확도는 미확인.


### 실패별 개선안과 검증 기준 (미구현 제안)

| 실패/현재 산출물 | 검토할 개선안 | 개선으로 인정할 관찰 기준 | 현재 상태 |
|---|---|---|---|
| 높이: 색인 내 직접 근거 없음 | 출처/이용 조건이 확인된 높이 문서 범위 확보 | 장소/높이/단위를 직접 지지하는 ID와 원문 확인 후 gold 지정 | 교수 파일 미제공, 추가 수집/색인 미실시 |
| 기후: D019후보 포함, D011선택 | 질문 속성과 문장 관련성을 별도로 점검하는 선택 기준 | 동일 문서/k/생성 조건에서 D019 등 직접 기후 문장이 실제 입력으로 전달됨 | 제공 알고리즘 유지, 개선 구현 미실시 |
| 높이1000m/기후서부남부: 전달 근거 밖 생성 | 답을 못 지지하는 문맥에서 유보하도록 관찰 기준 설정 | 무관한 근거일 때 '근거 부족', 관련 근거일 때 직접 지지되는 사실만 출력 | 실제3출력에서는 유보 실패 확인, 개선 검증 미실시 |
| 최신 운영 시간: 정적 개요로 답 불가 | 확인된 특정 관광지/기준일 공지와 유효성 조건 마련 | 최신 공식 공지 없으면 유보, 있으면 시각/출처 일치 | 최신 자료/생성 미측정, 임의 운영 시간 보충 없음 |

- 실행 전 예측 시점의 별도 학생 기록은 미수신. 실행 후 구조 설명: 단어는 조사 포함 전체 어휘가 없어 후보0개, 문자는 부분 n-gram 공유로 후보 생성 가능. 이를 실제 학생의 사전 예측으로 소급 기록하지 않음.
